# Isolation Forest Anomaly Detection for TLS Profiling

This notebook evaluates one-class Isolation Forest baselines on extracted TLS traffic features. For each target label, the detector is trained only on flows from that label and then evaluated on its ability to separate that label from the rest of the traffic.


In [6]:
import sys
sys.path.append('../../src')

## Feature Groups

- `FLOW`: basic bidirectional flow statistics such as bytes, packets, rates, and duration (`bs`, `ps`, `br`, `pr`, `td`)
- `CTLS`: client-side TLS metadata (`tls.cver`, `tls.ccs`, `tls.cext`, `tls.csg`, `tls.alpn`, `tls.csv`)
- `STLS`: server-side TLS metadata (`tls.sver`, `tls.scs`, `tls.sext`, `tls.ssv`)
- `REC`: ordered sequence of signed TLS record lengths (`tls.rec`, first 20 records)

The notebook evaluates each group individually and all group combinations to understand which feature families carry the most discriminative signal.


In [7]:
FLOW = ["bs","ps", "br", "pr", "td"]
CTLS = ["tls.cver", "tls.ccs", "tls.cext", "tls.csg", "tls.alpn", "tls.csv"]
STLS = ["tls.sver", "tls.scs", "tls.sext", "tls.ssv"]
REC = ["tls.rec"]

In [8]:
from pathlib import Path

import joblib
import pandas as pd

data_dir = Path("../data-ext")
#data_dir = Path("../data")

print(f"Loading extracted features from {data_dir}.")
df_train = pd.read_parquet(data_dir / "malware_train.parquet")
df_val = pd.read_parquet(data_dir / "malware_val.parquet")
df_test = pd.read_parquet(data_dir / "malware_test.parquet")
df_train_label = pd.read_parquet(data_dir / "malware_train_labels.parquet")
df_val_label = pd.read_parquet(data_dir / "malware_val_labels.parquet")
df_test_label = pd.read_parquet(data_dir / "malware_test_labels.parquet")

preprocessor_path = data_dir / "malware_preprocessor.joblib"
if not preprocessor_path.exists():
    fallback_path = data_dir / "preprocessor.joblib"
    if not fallback_path.exists():
        raise FileNotFoundError(
            f"Could not find {preprocessor_path} or {fallback_path}. Run the data preparation notebook first."
        )
    print(f"{preprocessor_path.name} not found, using {fallback_path.name} instead.")
    preprocessor_path = fallback_path

preprocessor = joblib.load(preprocessor_path)
print(f"Loaded preprocessor from {preprocessor_path}.")

Loading extracted features from ../data-ext.
Loaded preprocessor from ../data-ext/malware_preprocessor.joblib.


In [ ]:
from tls_profiling.baselines.model_isolation_forest import IsolationForestDetector, Config
from typing import Any, Dict
import numpy as np

def train_detector(train:np.ndarray) -> IsolationForestDetector:
    detector=IsolationForestDetector(Config())
    detector.fit(train)
    return detector

from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

import warnings

warnings.filterwarnings(
    "ignore",
    message=r"unknown class\(es\) .* will be ignored",
    category=UserWarning,
    module=r"sklearn\.preprocessing\._label",
)

def select_feature_set(x, feature_set):
    if not feature_set:
        return x

    selected_columns = [
        column for column in x.columns
        if any(column.startswith(prefix) for prefix in feature_set)
    ]
    return x.loc[:, selected_columns]

def choose_f1_threshold(y_true, anomaly_score):
    precision, recall, thresholds = precision_recall_curve(y_true, anomaly_score)

    if len(thresholds) == 0:
        return float("inf")

    f1_scores = (2 * precision[:-1] * recall[:-1]) / np.clip(
        precision[:-1] + recall[:-1],
        a_min=1e-12,
        a_max=None,
    )
    best_idx = int(np.nanargmax(f1_scores))
    return float(thresholds[best_idx])

def evaluate(feature_set, normal_label):
    # x_train: only normal traffic
    x_train = df_train.loc[df_train_label["connection_label"] == normal_label].reset_index(drop=True)
    # x_val: mixed traffic used to tune the F1 threshold
    x_val = df_val
    y_val = (df_val_label["connection_label"] != normal_label).astype(int)
    # y_test: 1 = anomaly, 0 = normal
    y_test = (df_test_label["connection_label"] != normal_label).astype(int)
    x_test = df_test

    # prepare datasets:
    x_train_transformed = select_feature_set(preprocessor.transform(x_train), feature_set)
    x_val_transformed = select_feature_set(preprocessor.transform(x_val), feature_set)
    x_test_transformed = select_feature_set(preprocessor.transform(x_test), feature_set)

    # create detector on TRAIN:
    detector = train_detector(x_train_transformed)
    
    # choose the F1 threshold on the mixed validation split
    val_scores = detector.score(x_val_transformed)
    threshold = choose_f1_threshold(y_val, val_scores)
    
    # evaluate on TEST:
    anomaly_score = detector.score(x_test_transformed)

    return y_test, anomaly_score, threshold

def debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score):
    """
    Write the intermediate data to CSV file.
    """
    # store y_test, y_pred to CSV
    feature_set_name = "_".join(feature_set)
    class_label_name = normal_label

    output_path = f"tmp/if_{class_label_name}_{feature_set_name}.csv"
    pd.DataFrame({
        "y_test": y_test,
        "y_pred": y_pred,
        "anomaly_score": anomaly_score,
    }).to_csv(output_path, index=False)
    
def compute_scores(feature_set, normal_label):
    y_test, anomaly_score, threshold = evaluate(feature_set, normal_label)
    # y_true = ground truth labels; y_pred = thresholded anomaly decisions
    auc = roc_auc_score(y_test, anomaly_score)
    ap = average_precision_score(y_test, anomaly_score)
    y_pred = (anomaly_score >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred)   # TRUE ~ 1, FALSE ~ 0

    debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score)
    return { "auc_score": auc, "ap_score" : ap, "f1_score" : f1 }

def plot_curves(feature_set, normal_label):
    y_test,anomaly_score,threshold = evaluate(feature_set, normal_label)

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    PrecisionRecallDisplay.from_predictions(
        y_test,
        anomaly_score,
        name="IsolationForest PR-AUC",
        ax=axes[0],
    )
    axes[0].set_title(f"{normal_label} Precision-Recall")

    RocCurveDisplay.from_predictions(
        y_test,
        anomaly_score,
        name="IsolationForest AUC-ROC",
        ax=axes[1],
    )
    axes[1].set_title(f"{normal_label} ROC Curve")

    plt.tight_layout()
    plt.show()


## Evaluation

For each label in `system`, `unknown`, and `malware`, that label is treated as the in-profile or normal class. All remaining labels are treated as anomalies.

The evaluation uses three disjoint splits:

- `train`: only samples from the selected normal class, used to fit the Isolation Forest
- `validation`: mixed traffic, used to choose the anomaly-score threshold that maximizes `F1`
- `test`: held-out mixed traffic, used only for final reporting

Feature ablation:

- all single groups and all group combinations built from `FLOW`, `CTLS`, `STLS`, and `REC`

Reported metrics:

- `AUC-ROC`: how well the raw anomaly scores rank the selected class against the rest over all thresholds
- `AP`: average precision, which is especially useful when the class balance is uneven
- `F1`: test-set score obtained with the threshold selected on the validation split

When reading the tables below, `AUC-ROC` and `AP` describe score quality, while `F1` describes one concrete operating point after threshold selection.


In [10]:
from itertools import combinations

feature_groups = {
    "FLOW": FLOW,
    "CTLS": CTLS,
    "STLS": STLS,
    "REC": REC,
}

import pandas as pd
from itertools import combinations

rows = []

group_names = list(feature_groups)
for size in range(1, len(group_names) + 1):
    for group_combo in combinations(group_names, size):
        selected_features = []
        for group_name in group_combo:
            selected_features.extend(feature_groups[group_name])

        feature_set_name = ", ".join(group_combo)

        for label in ["system", "unknown", "malware"]:
            scores = compute_scores(selected_features, label)

            rows.append({
                "FeatureSet": feature_set_name,
                "ClassLabel": label,
                "AucScore": scores["auc_score"],
                "ApScore": scores["ap_score"],
                "F1Score" : scores["f1_score"]
            })

results_df = pd.DataFrame(rows, columns=["FeatureSet", "ClassLabel", "AucScore", "ApScore", "F1Score"])
display(results_df)

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
0,FLOW,system,0.915160,0.915633,0.912966
1,FLOW,unknown,0.445438,0.830989,0.928980
2,FLOW,malware,0.544079,0.532410,0.679484
3,CTLS,system,0.933748,0.951822,0.915906
4,CTLS,unknown,0.541046,0.883740,0.928980
5,CTLS,malware,0.261428,0.480208,0.707669
6,STLS,system,0.953219,0.943988,0.968401
7,STLS,unknown,0.728589,0.922834,0.954673
8,STLS,malware,0.819231,0.800379,0.826509
9,REC,system,0.834621,0.874346,0.858966


## System Profile

The `system` profile is the most stable and the easiest to model. Across most feature sets, both `AUC-ROC` and `AP` are high, which means the detector can rank system traffic against the rest with relatively little ambiguity.

The strongest results are concentrated around TLS metadata and record-level features:

- best `AP`: `CTLS, REC` with approximately `0.9729`
- best `AUC-ROC`: `FLOW, CTLS, STLS` with approximately `0.9699`
- best `F1`: `STLS` alone with approximately `0.9684`, with several multi-group combinations very close behind

This suggests that system traffic exposes a coherent and repeatable structure, especially on the TLS side. Server-side TLS features are already very strong on their own, while combining them with client TLS or record-sequence information improves the ranking metrics even further.


In [16]:
system_df = results_df[results_df["ClassLabel"] == "system"].sort_values("F1Score", ascending=False)
display(system_df)

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
6,STLS,system,0.953219,0.943988,0.968401
15,"FLOW, STLS",system,0.953485,0.947961,0.967792
42,"FLOW, CTLS, STLS, REC",system,0.964123,0.965235,0.959764
21,"CTLS, STLS",system,0.957445,0.966874,0.958106
30,"FLOW, CTLS, STLS",system,0.969875,0.972283,0.956386
36,"FLOW, STLS, REC",system,0.952248,0.952080,0.953312
33,"FLOW, CTLS, REC",system,0.965488,0.966402,0.953247
39,"CTLS, STLS, REC",system,0.959808,0.967573,0.944961
24,"CTLS, REC",system,0.965127,0.972900,0.942528
12,"FLOW, CTLS",system,0.966842,0.970852,0.940952


## Malware Profile

The `malware` profile is noticeably harder. Compared with `system`, the scores vary much more across feature sets, which indicates that malware traffic is less homogeneous and overlaps more with the rest of the dataset.

A clear pattern in the current results is that the best feature sets consistently include server-side TLS metadata and, often, the TLS record sequence:

- best overall configuration: `STLS, REC` with approximately `AUC-ROC = 0.8903`, `AP = 0.8470`, and `F1 = 0.8788`
- `STLS` alone is already strong, with `AP` around `0.8004`
- feature sets dominated by `CTLS` perform much worse, which suggests that client-side TLS metadata is not stable enough to define a clean malware-only profile in this setup

A reasonable interpretation is that malware families may share server responses and record-level exchange patterns more consistently than they share one client-side handshake signature. That makes `STLS` and `REC` much more useful than `FLOW` or `CTLS` for this profile.


In [18]:
malware_df = results_df[results_df["ClassLabel"] == "malware"].sort_values("F1Score", ascending=False)
display(malware_df)

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
29,"STLS, REC",malware,0.890295,0.846986,0.878815
17,"FLOW, STLS",malware,0.867450,0.779408,0.845870
38,"FLOW, STLS, REC",malware,0.836734,0.771226,0.830496
8,STLS,malware,0.819231,0.800379,0.826509
44,"FLOW, CTLS, STLS, REC",malware,0.685401,0.576132,0.782868
41,"CTLS, STLS, REC",malware,0.658127,0.565242,0.772938
32,"FLOW, CTLS, STLS",malware,0.576616,0.500511,0.726135
35,"FLOW, CTLS, REC",malware,0.554821,0.494530,0.723852
26,"CTLS, REC",malware,0.555864,0.499704,0.714752
11,REC,malware,0.736756,0.721496,0.712228


## Unknown Category

The `unknown` label should be interpreted carefully. It is not a semantic traffic type like `system` or `malware`; it is mainly a residual bucket for traffic that was not confidently assigned elsewhere.

Because of that, strong threshold-based metrics do not automatically mean that we learned a clean and meaningful `unknown` profile. In this setup, the anomaly class for the `unknown` experiment is very broad because it contains both `system` and `malware` traffic.

That pattern is visible in the results:

- best `AP`: `STLS` with approximately `0.9228`
- best `F1`: also driven by `STLS`-based feature sets, reaching approximately `0.9547`
- however, the best `AUC-ROC` is only about `0.7286`, which is much lower than for the `system` profile

So the model can still find a useful decision threshold, but the raw score ordering is much less clean. This is consistent with `unknown` being heterogeneous rather than a single well-defined behavior class. In other words, these results are useful as a reference point, but they should not be interpreted as evidence of a strong standalone `unknown` profile.


In [17]:
unknown_df = results_df[results_df["ClassLabel"] == "unknown"].sort_values("F1Score", ascending=False)
display(unknown_df)

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
7,STLS,unknown,0.728589,0.922834,0.954673
22,"CTLS, STLS",unknown,0.649442,0.899996,0.954392
31,"FLOW, CTLS, STLS",unknown,0.632251,0.887526,0.949659
43,"FLOW, CTLS, STLS, REC",unknown,0.621113,0.895020,0.938490
34,"FLOW, CTLS, REC",unknown,0.564080,0.883131,0.938338
28,"STLS, REC",unknown,0.641505,0.912410,0.936527
25,"CTLS, REC",unknown,0.595267,0.897167,0.936526
13,"FLOW, CTLS",unknown,0.506157,0.871985,0.935815
40,"CTLS, STLS, REC",unknown,0.640380,0.902872,0.933339
10,REC,unknown,0.535931,0.870843,0.931567
